In [1]:
import pandas as pd
import os

import sys
sys.path.append('..')
from helpers import get_dst_timestamps

In [2]:
nyiso_zone_subba_map = {
    'WEST': 'ZONA',
    'GENESE': 'ZONB',
    'CENTRL': 'ZONC',
    'NORTH': 'ZOND',
    'MHK VL': 'ZONE',
    'CAPITL': 'ZONF',
    'HUD VL': 'ZONG',
    'MILLWD': 'ZONH',
    'DUNWOD': 'ZONI',
    'N.Y.C.': 'ZONJ',
    'LONGIL': 'ZONK'
}

In [3]:
# Profiles downloaded from https://www.nyiso.com/custom-reports
load_profile_list = []
forecast_profile_list = []

dir = '../data/iso_load_profiles/raw/nyiso'
for filename in os.listdir(dir):
    f = os.path.join(dir, filename)
    df = pd.read_csv(f)

    zone = df.loc[0, 'Zone Name']
    year = int(df.loc[0, 'Eastern Date Hour'].split('/')[0])
    forecast = 'Forecast' in filename

    df['subba'] = df['Zone Name'].map(nyiso_zone_subba_map)
    df['timestamp'] = (
        pd.to_datetime(df['Eastern Date Hour']) + pd.Timedelta(hours=1)
    )

    # Convert from daylight to standard time
    dst_start, dst_end = get_dst_timestamps(year)
    df.loc[(
        (df.timestamp > dst_start) & (df.timestamp <= dst_end)
    ), 'timestamp'] -= pd.Timedelta(hours=1)
    df.loc[(
        df.duplicated(subset=['timestamp', 'subba'], keep='last')
    ), 'timestamp'] = dst_end

    # Convert from EST to UTC
    df['timestamp'] += pd.Timedelta(hours=5)

    if forecast:
        df = (
            df.rename(columns={'DAM Forecast Load': 'value'})
            [['timestamp', 'subba', 'value']]
        )
        forecast_profile_list.append(df)
    else:
        df = (
            df.rename(columns={'TWI Actual Load': 'value'})
            [['timestamp', 'subba', 'value']]
        )
        load_profile_list.append(df)

In [4]:
nyiso_load = (
    pd.concat(load_profile_list, ignore_index=True)
    .drop_duplicates(keep='first')
)
nyiso_load.to_csv(f"../data/iso_load_profiles/nyiso.csv", index=False)

nyiso_forecast = (
    pd.concat(forecast_profile_list, ignore_index=True)
    .drop_duplicates(keep='first')
)
nyiso_forecast.to_csv(f"../data/iso_load_profiles/nyiso_forecast.csv", index=False)